In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 10

df = pd.read_csv('../datasets/master/master_final_2025.csv')

In [2]:
quad_colors = {
    'High Access - High Outcome': '#2E7D32',
    'High Access - Low Outcome (masalah kualitas)': '#E65100',
    'Low Access - High Outcome (efisien)': '#1565C0',
    'Low Access - Low Outcome (butuh infrastruktur)': '#B71C1C',
}

In [3]:
# ============ CHART 1: Quadrant scatter (access vs outcome) ============
fig, ax = plt.subplots(figsize=(11,9))
for q, c in quad_colors.items():
    sub = df[df['quadrant']==q]
    ax.scatter(sub['access_index'], sub['outcome_index_used'], c=c, label=q, s=90, edgecolor='white', linewidth=0.8, zorder=3)
 
for _, r in df.iterrows():
    label = r['province'].title()
    ax.annotate(label, (r['access_index'], r['outcome_index_used']), fontsize=7.5,
                xytext=(4,4), textcoords='offset points', alpha=0.85)
 
ax.axhline(df['outcome_index_used'].median(), color='grey', ls='--', lw=0.8, zorder=1)
ax.axvline(df['access_index'].median(), color='grey', ls='--', lw=0.8, zorder=1)
ax.set_xlabel('Access Index (ketersediaan fasilitas: rasio desa berSD/SMP/SMA + kepadatan sekolah)')
ax.set_ylabel('Outcome Index (literacy + EYS + completion SMA, distandarisasi)')
ax.set_title('Kuadran Akses vs Outcome Pendidikan per Provinsi\n(garis putus-putus = median)', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=8.5, framealpha=0.95)
ax.text(0.99,0.01,'Note: 4 provinsi pemekaran Papua pakai outcome index parsial (literacy+EYS saja, data completion SMA belum tersedia)',
        transform=ax.transAxes, fontsize=7, ha='right', style='italic', color='grey')
plt.tight_layout()
plt.savefig('../src/visualization/01_quadrant_access_outcome.png', dpi=150)
plt.close()

In [4]:
# ============ CHART 2: Access-Outcome GAP ranked bar ============
fig, ax = plt.subplots(figsize=(10,10))
d = df.sort_values('access_outcome_gap')
colors = ['#B71C1C' if x<0 else '#E65100' for x in d['access_outcome_gap']]
ax.barh(d['province'].str.title(), d['access_outcome_gap'], color=colors)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Access-Outcome Gap (Access Index − Outcome Index)')
ax.set_title('Kesenjangan Akses vs Outcome per Provinsi\n(oranye = akses lebih baik drpd outcome / masalah kualitas;\nmerah = outcome relatif lebih baik drpd akses)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../src/visualization/02_access_outcome_gap.png', dpi=150)
plt.close()

In [5]:
# ============ CHART 3: Gender gap by region ============
fig, axes = plt.subplots(1,2, figsize=(13,5.5))
region_order = df.groupby('region')['gap_literacy'].mean().sort_values().index
sns_data = df.groupby('region')[['gap_literacy','gap_eys']].mean().reindex(region_order)
 
axes[0].barh(sns_data.index, sns_data['gap_literacy'], color='#6A1B9A')
axes[0].axvline(0, color='black', lw=0.8)
axes[0].set_title('Rata-rata Gender Gap: Literacy Rate\n(perempuan − laki-laki, poin persen)')
axes[0].set_xlabel('Gap (pp)')
 
axes[1].barh(sns_data.index, sns_data['gap_eys'], color='#00838F')
axes[1].axvline(0, color='black', lw=0.8)
axes[1].set_title('Rata-rata Gender Gap: Expected Years of Schooling\n(perempuan − laki-laki, tahun)')
axes[1].set_xlabel('Gap (tahun)')
plt.suptitle('Kesenjangan Gender dalam Pendidikan per Wilayah', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../src/visualization/03_gender_gap_by_region.png', dpi=150)
plt.close()

In [6]:
# ============ CHART 4: Library/reading vs outcome (no correlation finding) ============
fig, axes = plt.subplots(1,2, figsize=(13,5.5))
sub = df.dropna(subset=['completion_senior_high'])
axes[0].scatter(sub['library_per_1000pupils'], sub['completion_senior_high'], c='#1565C0', s=70, alpha=0.8)
z = np.polyfit(sub['library_per_1000pupils'], sub['completion_senior_high'], 1)
xs = np.linspace(sub['library_per_1000pupils'].min(), sub['library_per_1000pupils'].max(),50)
axes[0].plot(xs, np.poly1d(z)(xs), 'r--', lw=1.5)
axes[0].set_xlabel('Jumlah perpustakaan per 1000 murid SD')
axes[0].set_ylabel('Completion Rate SMA (%)')
axes[0].set_title(f'r = {sub["library_per_1000pupils"].corr(sub["completion_senior_high"]):.2f} (tidak signifikan)')
 
sub2 = df.dropna(subset=['reading_fondness_level','completion_senior_high'])
axes[1].scatter(sub2['reading_fondness_level'], sub2['completion_senior_high'], c='#E65100', s=70, alpha=0.8)
z2 = np.polyfit(sub2['reading_fondness_level'], sub2['completion_senior_high'], 1)
xs2 = np.linspace(sub2['reading_fondness_level'].min(), sub2['reading_fondness_level'].max(),50)
axes[1].plot(xs2, np.poly1d(z2)(xs2), 'r--', lw=1.5)
axes[1].set_xlabel('Indeks Kegemaran Membaca (TGM)')
axes[1].set_ylabel('Completion Rate SMA (%)')
axes[1].set_title(f'r = {sub2["reading_fondness_level"].corr(sub2["completion_senior_high"]):.2f} (tidak signifikan)')
 
plt.suptitle('Akses Perpustakaan & Minat Baca vs Completion Rate SMA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../src/visualization/04_library_reading_vs_outcome.png', dpi=150)
plt.close()

In [9]:
quad_colors = {
    'High Access - High Outcome': '#2E7D32',
    'High Access - Low Outcome (masalah kualitas)': '#E65100',
    'Low Access - High Outcome (efisien)': '#1565C0',
    'Low Access - Low Outcome (butuh infrastruktur)': '#B71C1C',
}

fig, axes = plt.subplots(1,2, figsize=(17,8.5))

for ax, zoom in zip(axes, [False, True]):
    d = df.copy()
    if zoom:
        d = d[~d['province'].isin(['PAPUA TENGAH','PAPUA PEGUNUNGAN'])]
    for q, c in quad_colors.items():
        sub = d[d['quadrant']==q]
        ax.scatter(sub['access_index'], sub['outcome_index_used'], c=c, label=q, s=100, edgecolor='white', linewidth=0.8, zorder=3)
    for _, r in d.iterrows():
        ax.annotate(r['province'].title(), (r['access_index'], r['outcome_index_used']), fontsize=8,
                    xytext=(4,4), textcoords='offset points', alpha=0.85)
    ax.axhline(df['outcome_index_used'].median(), color='grey', ls='--', lw=0.8, zorder=1)
    ax.axvline(df['access_index'].median(), color='grey', ls='--', lw=0.8, zorder=1)
    ax.set_xlabel('Access Index (rasio desa berSD/SMP/SMA + densitas sekolah)')
    ax.set_ylabel('Outcome Index (literacy + EYS + completion SMA, standardized)')
    ax.set_title('Zoom (tanpa Papua Tengah & Pegunungan)' if zoom else 'Semua 38 Provinsi', fontsize=12, fontweight='bold')
    if not zoom:
        ax.legend(loc='lower right', fontsize=8.5, framealpha=0.95)

plt.suptitle('Kuadran Akses vs Outcome Pendidikan per Provinsi', fontsize=14, fontweight='bold')
fig.text(0.5, 0.005, 'Note: Papua Tengah & Papua Pegunungan memang punya literacy rate riil terendah nasional (68-86%) — bukan artefak data hilang. 4 provinsi pemekaran Papua pakai outcome index parsial (completion SMA belum tersedia).',
        ha='center', fontsize=8.5, style='italic', color='grey')
plt.tight_layout(rect=[0,0.02,1,1])
plt.savefig('../src/visualization/01_quadrant_access_outcome.png', dpi=150)
plt.close()
print("done")

done
